# EA2 — Despliegue y gobierno de una infraestructura de datos en la nube

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | *23* |
| **Integrantes** | *Manuela Tangarife Zapata* |
| **Caso de estudio** | *Wanderbricks* |
| **Fecha de entrega** | domingo 6 de septiembre |
| **🎥 Enlace al video** | https://drive.google.com/file/d/1Z-cbxg9HpztWpBKjYNFv5G-gNGTVvuJZ/view?usp=sharing |

---
## 1. Contexto y problema

Wanderbircks es un marketplace de alquiler vacacional que maneja información sobre usuarios, anfitriones, propiedades, reservas, pagos, reseñas y comportamiento de navegación. Para analizar esta información no basta solo con tener los datos. También se necesita un entorno que permita almacenarlos, procesarlos y administrarlos de forma organizada. 

El entorno debe dejar trabajar con distintos tipos de datos. Debe mantener una separación entre los datos originales y los datos transformados. Además tiene que facilitar el acceso según las responsabilidades de cada usuario. También debe permitir que los procesos se automaticen para evitar que las actualizaciones dependan solo de una ejecución manual. 

Por esta razón, propongo un entorno en Databricks organizado en capas Bronze, Silver y Gold. En este entorno, los datos pasan desde su estado original hasta convertirse en información lista para el análisis. Tambien se tienen en cuenta los permisos, el gobierno, el linaje y la automatización, para que la solución se mantenga y crezca con el tiempo.

---
## 2. Descripción de los datos

En la actividad anterior se trabajó con nueve tablas del caso Wanderbricks: bookings, payments, users, properties, hosts, destinations, reviews, clickstream y customer_support_logs. Estas tablas en conjunto contienen 545.676 registros y representan información de reservas, pagos, usuarios, propiedades, destinos, reseñas, navegación y soporte. 

Los datos son principalmente estructurados, aunque también hay elementos semiestructurados, como los campos anidados de clickstream y los mensajes almacenados en customer_support_logs. Algunas tablas también tienen texto libre, por ejemplo las reseñas y los registros de soporte.

En esta actividad planteo una actualización periódica del entorno, mediante una programación diaria para los procesos automatizados. Los principales usuarios serían los analistas, los ingenieros de datos y los administradores del entorno.

---
## 3. Decisiones de diseño y justificación
### 3.1 Diagrama de la arquitectura

![Diagrama EA2.drawio_1789931265390.png](./Diagrama EA2.drawio_1789931265390.png "Diagrama EA2.drawio_1789931265390.png")

En esta arquitectura, Databricks gestiona la infraestructura y los componentes adminitrados de la plataforma, y el equipo define la organización lógica de los datos, las transformaciones, los permisos, procesos y automatizaciones.

En la capa Bronze se guardan los datos originales, en Silver se hacen las transformaciones y valididaciones necesarias y en Gold se generan datos listos para el análisis. Los datos pueden ser consultados mediante SQL y PySpark, y con Catalog Explorer se pueden administrar y consultar elementos del entorno.

### 3.2 Matriz de roles

Los permisos se plantean en función de cada rol. El analista debe poder consultar la información para hacer sus análisis, pero no debe poder modificar los datos originales. El ingeniero de datos tiene que trabajar con los procesos de transformación y mantenimiento de las tablas. El administrador es responsable de la seguridad, los permisos y el gobierno del entorno.

| Rol | Bronce | Plata | Oro |
|---|---|---|---|
| Analista | Lectura | Lectura | Lectura |
| Ingeniero de datos | Lectura y escritura | Lectura y escritura | Creación y mantenimiento |
| Administrador | Gestión completa | Gestión completa | Gestión Completa |

### 3.3 Especificación del equivalente IaaS

Como equivalente de laaS se plantea la ejecución de la solución en máquinas virtuales que administradas.

La idea es una máquina para la orquestación de procesos utilizando Airflow, una máquina principal para Spark y dos máquinas worker para distribuir el procesamiento. Se trabajaría sobre el sistema operativo Ubuntu Server LTS. 

La máquina principal podría tener alrededor de 4 vCPU y 16 GB de memoria, mientras que cada worker tendría cerca de 8 vCPU y 32 GB de memoria. Se contempla un espacio de almacenamiento inicial de 500 GB, con posibilidad de ampliación en función del crecimiento de los datos. 

La infraestructura estaría dentro de una red privada, con acceso administrativo restringido y reglas de seguridad para separar el tráfico interno del acceso externo. En las máquinas habría que instalar y configurar Spark, Python, Airflow y las dependencias necesarias para ejecutar los procesos.

### 3.4 Comparación IaaS / PaaS / SaaS

| Criterio | IaaS | PaaS | SaaS |
|---|---|---|---|
| Control | Alto | Medio | Bajo |
| Tiempo hasta el primer resultado | Mayor | Menor | Muy bajo |
| Esfuerzo operativo | Alto | Medio o bajo | Bajo |
| Costo | Depende de la infrestructura y operación | Depende del uso de la plataforma | Normalmente por suscripción o uso |
| Escalabilidad | Requiere configuración | Facilitada por la plataforma | Principalmente administrada por el proveedor |
| Gobierno |Mayor control, pero mayor responsabilidad | Control centralizado con servicios administrados | Depende de las opciones del producto|

**Conclusión:** Para el caso de Wanderbricks considero conveniente utilizar una plaataforma administrada como Databricks, ya que permite trabajar con los datos y los procesos sin tener que administrar directamente las máquinas, sistema operativo y gran parte de la infraestructura. Esto permite concentración en la organización de los datos, las transformaciones, los permisos y los procesos analíticos.

El enfoque laaS tendría sentido si se necesitara un mayor control sobre la infraestructura y la configuración de las máquinas. Por otro lado un modelo SaaS estaría más orientado al consumo de una aplicación ya terminada y ofrecería menos control sobre la infraestructura y los procesos internos.

---
## 4. Implementación
### 4.1 Organización del entorno

In [0]:
# Reviso la organización actual del entorno

CATALOGO = "bigdata_grupo23"
ESQUEMA = "wanderbricks"

print("Catalogo utilizado:", CATALOGO)
print("Esquema utilizado:", ESQUEMA)

# Consulto los esquemas disponibles dentro del catálogo
print("Esquemas disponibles:")
display(spark.sql(f"SHOW SCHEMAS IN {CATALOGO}"))

# Consulto los volúmenes disponibles dentro del esquema
print("Volúmenes disponibles:")
display(spark.sql(f"SHOW VOLUMES IN {CATALOGO}.{ESQUEMA}"))

# Consulto las tablas disponibles dentro del esquema
print("Tablas del esquema:")
display(spark.sql(f"SHOW TABLES IN {CATALOGO}.{ESQUEMA}"))

### 4.2 Permisos

In [0]:
# Primero revisar los grupos disponibles en Databricks.
# Necesito sus nombres exactos para poder hacer los GRANT.

display(spark.sql("SHOW GROUPS"))

In [0]:
# GRANT 1 — a quién, qué y por qué:
# Doy permiso de lectura sobre la tabla Gold
spark.sql("""
GRANT SELECT
ON TABLE bigdata_grupo23.wanderbricks.destinations_gold
TO `manuela.tangarife@est.iudigital.edu.co`	
""")

print("GRANT 1 ejecutado correctamente")

# GRANT 2 — a quién, qué y por qué:
# Doy permiso para modificar la tabla Bronze
spark.sql("""
GRANT MODIFY 
ON TABLE bigdata_grupo23.wanderbricks.bookings_bronze
TO `manuela.tangarife@est.iudigital.edu.co`
""")

print("GRANT 2 ejecutado correctamente.")

# display(spark.sql(f"SHOW GRANTS ON TABLE {TABLA}"))
# Verifico los permisos asignados sobre la tabla Gold
print("Permisos de destinations_gold:")
display(
    spark.sql("""
    SHOW GRANTS ON TABLE bigdata_grupo23.wanderbricks.destinations_gold
    """)
)

# Verifico los permisos asignados sobre la tabla Bronze
print("Permisos de bookings_bronze:")
display(
    spark.sql("""
    SHOW GRANTS ON TABLE bigdata_grupo23.wanderbricks.bookings_bronze
    """)
)

### 4.3 Linaje

![image_1789940981439.png](./image_1789940981439.png "image_1789940981439.png")

In [0]:
spark.sql("""
DESCRIBE TABLE bigdata_grupo23.wanderbricks.destinations_gold
""").show(truncate=False)

destinations_gold.printSchema()

In [0]:
spark.sql("""
DROP TABLE IF EXISTS bigdata_grupo23.wanderbricks.destinations_gold
""")

print("Tabla destinations_gold anterior eliminada correctamente.")

In [0]:
from pyspark.sql import functions as F

bookings = spark.table("bigdata_grupo23.wanderbricks.bookings_silver")
properties = spark.table("bigdata_grupo23.wanderbricks.properties_silver")
destinations = spark.table("bigdata_grupo23.wanderbricks.destinations_silver")

# Uno las reservas, propiedades y destinos para obtener la información.
destinations_gold = (
    bookings
    #Excluyo las reservas que estan en estado de revición
    .filter(F.col("status") != "revision")
    
    # Relaciono cada reserva con su propiedad
    .join(
        properties,
        bookings.property_id == properties.property_id,
        "inner"
    )
    
    # Relaciono cada propiedad con su destino
    .join(
        destinations,
        properties.destination_id == destinations.destination_id,
        "inner"
    )
    
    # Agrupo los resultados por destino y país
    .groupBy(
        destinations.destination,
        destinations.country
    )
    
    # Calculo las principales métricas de las reservas 
    .agg(
        F.count(bookings.booking_id).alias("total_reservas"),
        F.round(F.sum(bookings.total_amount), 2).alias("valor_total_reservas"),
        F.round(F.avg(bookings.total_amount), 2).alias("valor_promedio_reserva")
    )
    # Fecha y hora de actualizacion de la tabla
    .withColumn("actualizado_en", F.current_timestamp())
)

# Guardo el resultado como una tabla Delta y reemplazo la versicón anterior.
destinations_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "bigdata_grupo23.wanderbricks.destinations_gold"
    )

print("Tabla Gold creada correctamente.")

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM bigdata_grupo23.wanderbricks.destinations_gold
    ORDER BY valor_total_reservas DESC
    """)
)

### 4.4 Automatización

In [0]:
from pyspark.sql import functions as F

# cargo las tablas Silver que voy a utilizar para construir la tabla Gold
bookings = spark.table("bigdata_grupo23.wanderbricks.bookings_silver")
properties = spark.table("bigdata_grupo23.wanderbricks.properties_silver")
destinations = spark.table("bigdata_grupo23.wanderbricks.destinations_silver")

# Se excluye revición porque fue usado como prueba Delta en la EA1.
destinations_gold = (
    bookings
    .filter(F.col("status") != "revision")
    .join(
        properties,
        bookings.property_id == properties.property_id,
        "inner"
    )
    .join(
        destinations,
        properties.destination_id == destinations.destination_id,
        "inner"
    )
    .groupBy(
        destinations.destination,
        destinations.country
    )
    .agg(
        F.count(bookings.booking_id).alias("total_reservas"),
        F.round(F.sum(bookings.total_amount), 2).alias("valor_total_reservas"),
        F.round(F.avg(bookings.total_amount), 2).alias("valor_promedio_reserva")
    )
        .withColumn("actualizado.en", F.current_timestamp())
)

# Guardo la información como una tabla Delta y reemplazo la versicón anterior
destinations_gold.write \
    .format("delta").option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(
        "bigdata_grupo23.wanderbricks.destinations_gold"
)

print("Gold actualizado correctamente.")

In [0]:
from pyspark.sql import functions as F

# Cargo la tabla Gold que voy a validar
gold = spark.table(
    "bigdata_grupo23.wanderbricks.destinations_gold"
)

# Cuento el total de registros de la tablas
total_filas = gold.count()

# Busco destinos duplicados según el nombre y el pais. 
duplicados = (
    gold
    .groupBy("destination", "country")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Cuento los registros que tienen datos nulos en destino o país 
nulos = gold.filter(
    F.col("destination").isNull() |
    F.col("country").isNull()
).count()

print("Total de filas:", total_filas)
print("Total de duplicados:", duplicados)
print("Filas con datos nulos:", nulos)

# Verifico que la tabla tenga registros, duplicados y que no existan datos nulos.
if total_filas == 0:
    raise ValueError("La tabla Gold no tiene registros.")
if duplicados > 0:
    raise ValueError("Se encontraron destinos duplicados.")
if nulos > 0:
    raise ValueError("Se encontraron valores nulos.")

print("Validación completada correctamente")

---
## 5. Resultados

Consulta 1 – Destinos con mayor valor de reservas

Phuket presenta el mayor valor total de reservas, con 7.292 reservas y un valor acumulado de 4.058.199,55. Le siguen Gold Coast y Mallorca. Los valores promedio por reserva son relativamente similares entre los principales destinos, por lo que las diferencias en el valor total están relacionadas principalmente con la cantidad de reservas.

Consulta 2 – Interacción y reservas por propiedad

Los resultados muestran que una mayor cantidad de interacciones no necesariamente se traduce en una mayor cantidad de reservas. Por ejemplo, la propiedad 4636 de Gold Coast registra 21 eventos de interacción, pero no presenta reservas. En cambio, la propiedad 16326 de Sardinia registra 15 interacciones y 6 reservas por un valor total de 3.642,66.

Consulta 3 – Métodos de pago más utilizados

Los cinco métodos de pago presentan cantidades de transacciones similares. credit_card registra el mayor número de pagos, con 9.168, y procesa aproximadamente 5.084.550,02. Por otro lado, bank_transfer presenta el mayor valor promedio por pago, con 560,57. En general, no se observa una diferencia muy marcada en el uso de los métodos de pago.

Consulta 4 – Calificación y cantidad de reservas

Las propiedades mostradas con mayor calificación presentan un promedio de 5,0, pero tienen diferentes cantidades de reservas. Además, en los resultados revisados cada propiedad cuenta solamente con una reseña. Por esta razón, los datos obtenidos no permiten establecer una relación clara entre una calificación alta y una mayor cantidad de reservas.

Consulta 5 – Dispositivos utilizados para navegar

La distribución de eventos entre dispositivos es bastante equilibrada. Las tablets registran 33.753 eventos (33,75 %), los computadores de escritorio 33.303 (33,30 %) y los dispositivos móviles 32.944 (32,94 %). La diferencia entre los tres dispositivos es pequeña, por lo que no se observa un predominio marcado de uno sobre los demás.

---
## 6. Conclusiones

**Qué funcionó**

Los datos obtenidos a través de las consultas fueron muy útiles para comparar las estadísticas de cada aspecto del negocio. Se pudieron observar tendencias significativas en cuanto a las reservas por destino en cuanto a cantidad y valor, mientras que las características de las reservas, como el método de pago o el dispositivo, presentaban una distribución más uniforme. Además, se pudo establecer una correlación entre las interacciones con la reserva y el número de opiniones de los alojamientos.


**Qué no funcionó o qué limitaciones se encontraron**

Durante el desarrollo fue necesario revisar algunos JOIN y la estructura de las tablas para evitar que la combinación de información a diferentes niveles de detalle alterara los resultados. En particular, para la consulta de calificaciones y reservas fue necesario calcular primero las métricas por propiedad y después realizar el JOIN.


También se me presentó una dificultad al configurar la automatización mediante Jobs, ya que fue necesario ajustar la ejecución de las tareas y verificar que el proceso completo funcionara correctamente. Esto mostró la importancia de probar por separado la construcción de la tabla Gold y su posterior validación antes de ejecutar el flujo automatizado.

Además, algunas propiedades con calificación de 5,0 tenían solamente una reseña, por lo que la calificación por sí sola no permite establecer una relación clara con la cantidad de reservas.

**¿Qué haría de forma diferente?**

En caso de volver a desarrollar el código, pondría mayor énfasis en analizar la granularidad de cada tabla y cómo los JOIN modificarían la generalización de los datos. También llevaría a cabo un análisis más detallado incluyendo más variables, ya que disponer de más información sobre las reservas y los alojamientos —como el número total de opiniones— podría resultar útil, aunque la correlación entre las dos variables no sea significativa.

---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| Manuela Tangarife Zapata | arquitectura, organización de datos, permisos, linaje, consultas, validación y automatización. | decisiones de arquitectura, los GRANT, el linaje, la construcción de la tabla Gold, los resultados, la validación y la automatización |

**Uso de asistentes de IA:** Copilot

---
## 📹 Preguntas obligatorias de sustentación

Cada integrante responde estas tres preguntas en su intervención del video:

1. ¿Qué parte de esta arquitectura administra el proveedor y cuál administran ustedes?
2. Muestre un GRANT que ejecutó y explique a quién le está dando qué, y por qué.
3. Si tuvieran que montar esto sobre máquinas virtuales, ¿qué sería lo primero que se les complicaría?

---
## ✅ Antes de entregar

- [ ] El diagrama de arquitectura está incluido y descrito
- [ ] Los dos GRANT están ejecutados y el SHOW GRANTS muestra el resultado
- [ ] El Job tiene dos o más tareas, está programado y hay evidencia de ejecución
- [ ] La comparación IaaS/PaaS/SaaS termina en una conclusión, no en una tabla suelta
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] Todo confirmado en /ea2 y el HTML subido a Canvas